In [ ]:
# On importe les outils dont on a besoin pour la carte
import os
import html
import webbrowser
from pathlib import Path
import folium
import pandas as pd
from folium.plugins import MarkerCluster, FastMarkerCluster

# Le chemin des dossiers qui contiennent les données
dossier = 'data/data/'        # le dossier Data
dossier_ref = 'ref/ref/'      # le dossier Ref

# Fonction qui lit le dossier Data
def lire(nom, base=dossier, cols=None):
    chemin = base + nom

    # On vérifie que le fichier existe si ont le trouve pas sa mais la phrase "Fichier introuvable"
    if not os.path.exists(chemin):
        raise FileNotFoundError(f"Fichier introuvable : {chemin} (dossier : {os.getcwd()})")
    
   # On lit le fichier, les colonnes sont séparées par des ";"
    try:
        d = pd.read_csv(chemin, sep=";", dtype=str, encoding="utf-8-sig",)
    except UnicodeDecodeError:

        # Si ça ne marche pas, on essaie un autre encodage
        d = pd.read_csv(chemin, sep=";", dtype=str, encoding="latin-1",)

    # On met les noms de colonnes en majuscules
    d.columns = d.columns.str.strip().str.upper()
    return d
    
# Fonction qui regroupe les valeurs dans un seul texte sans mettre de doublons
def joindre(serie):
    return ', '.join(sorted(set(serie.dropna().astype(str))))



# 1. SUPPORT

sup = lire('sup_support.txt')

# Les coordonnées de base du fichier sont en degrés, minutes, secondes 
# On met les coordonnées en nombre 
for c in ['COR_NB_DG_LAT', 'COR_NB_MN_LAT', 'COR_NB_SC_LAT',
          'COR_NB_DG_LON', 'COR_NB_MN_LON', 'COR_NB_SC_LON']:
    sup[c] = pd.to_numeric(sup[c], errors='coerce')

# Puis on les transforme en degrés décimal pour la carte
sup['lat'] = sup['COR_NB_DG_LAT'] + sup['COR_NB_MN_LAT'] / 60 + sup['COR_NB_SC_LAT'] / 3600

# La latitude Sud devient négative
sup.loc[sup['COR_CD_NS_LAT'] == 'S', 'lat'] *= -1

sup['lon'] = sup['COR_NB_DG_LON'] + sup['COR_NB_MN_LON'] / 60 + sup['COR_NB_SC_LON'] / 3600

# La longitude Ouest devient négative
sup.loc[sup['COR_CD_EW_LON'] == 'W', 'lon'] *= -1

# On enlève les lignes du fichier qui n'ont pas de coordonnées
sup = sup.dropna(subset=['lat', 'lon'])

# On demande un département / vide = toute la France
dep = input("Département (vide = toute la France) : ").strip().upper()
if dep:
    dep = dep.zfill(2)     # On transforme le 1 en 01

    # On garde seulement les supports de ce département
    sup = sup[sup['ADR_NM_CP'].str.startswith(dep, na=False)]

# On garde le lien entre chaque support et son installation
# (avant d'enlever les doublons, sinon on perd des opérateurs)
liens = sup[['SUP_ID', 'STA_NM_ANFR']].drop_duplicates()
ids_sup = set(sup['SUP_ID'])          # la liste des supports gardés
ids_sta = set(liens['STA_NM_ANFR'])   # la liste des installations gardées

# On enlève les supports qui sont en double
sup = sup.drop_duplicates(subset='SUP_ID')



# 2. NATURE ET PROPRIÉTAIRE

# On remplace les numéros par les vrais noms
sup = sup.merge(lire('sup_nature.txt', dossier_ref), on='NAT_ID', how='left')
sup = sup.merge(lire('sup_proprietaire.txt', dossier_ref), on='TPO_ID', how='left')



# 3. EXPLOITANTS ET DATE DE MISE EN SERVICE

station = lire('sup_station.txt')

# On garde seulement du dépardement choisi installations de notre département
station = station[station['STA_NM_ANFR'].isin(ids_sta)]

# On ajoute le nom de l'exploitant
station = station.merge(lire('sup_exploitant.txt', dossier_ref), on='ADM_ID', how='left')

# On transforme la date en vraie date
station['DATE'] = pd.to_datetime(station['DTE_EN_SERVICE'], dayfirst=True, errors='coerce')

# On relie chaque installation à son support
station = liens.merge(station[['STA_NM_ANFR', 'ADM_LB_NOM', 'DATE']],
                      on='STA_NM_ANFR', how='left')

# Pour chaque support ont mais ses exploitants et la date
infos_station = station.groupby('SUP_ID').agg(
    exploitants=('ADM_LB_NOM', joindre),
    mise_service=('DATE', 'min')
).reset_index()



# 4. LES ANTENNES

# On lit seulement les colonnes utiles
ant = lire('sup_antenne.txt', cols=['STA_NM_ANFR', 'AER_ID', 'TAE_ID', 'SUP_ID'])

# On garde seulement les antennes de nos supports
ant = ant[ant['SUP_ID'].isin(ids_sup)]

# On ajoute le type d'antenne
ant_t = ant.merge(lire('sup_type_antenne.txt', dossier_ref), on='TAE_ID', how='left')

# Pour chaque support ont mais le nombre d'antennes et leurs types
infos_ant = ant_t.groupby('SUP_ID').agg(
    nb_antennes=('AER_ID', 'nunique'),
    types_antenne=('TAE_LB', joindre)
).reset_index()



# 5. LES ÉMETTEURS

emr = lire('sup_emetteur.txt', cols=['STA_NM_ANFR', 'AER_ID', 'EMR_LB_SYSTEME'])

# On garde seulement les émetteurs de nos installations
emr = emr[emr['STA_NM_ANFR'].isin(ids_sta)]

# On relie chaque émetteur à son support
emr = emr.merge(ant[['STA_NM_ANFR', 'AER_ID', 'SUP_ID']],
                on=['STA_NM_ANFR', 'AER_ID'], how='inner')

# Pour chaque support ont mais la liste de ses systèmes
infos_emr = emr.groupby('SUP_ID').agg(systemes=('EMR_LB_SYSTEME', joindre)).reset_index()



# 6. ON RASSEMBLE TOUT

# On met toutes les informations dans un seul tableau
df = (sup.merge(infos_station, on='SUP_ID', how='left')
         .merge(infos_ant, on='SUP_ID', how='left')
         .merge(infos_emr, on='SUP_ID', how='left'))

# Fonction qui nettoie le texte, et met "?" si c'est vide
def txt(v, defaut='?'):
    return html.escape(str(v)) if pd.notna(v) and str(v).strip() else defaut

# Fonction qui crée le texte de la bulle d'un marqueur
def popup(r):

    # Si on n'a pas l'adresse, on prend la ligne d'adresse suivante
    adresse = r.ADR_LB_LIEU if pd.notna(r.ADR_LB_LIEU) else r.ADR_LB_ADD1

    # Si la date est inconnue, on met "?"
    date = r.mise_service.strftime('%d/%m/%Y') if pd.notna(r.mise_service) else '?'

    # La liste des information dans la bulle d'un marqueur
    return (
        f"<b>{txt(adresse, 'Support')}</b><br>"
        f"{txt(r.ADR_NM_CP, '')}<hr style='margin:4px 0'>"
        f"<b>Nature :</b> {txt(r.NAT_LB_NOM)}<br>"
        f"<b>Hauteur :</b> {txt(r.SUP_NM_HAUT)} m<br>"
        f"<b>Propriétaire :</b> {txt(r.TPO_LB)}<br>"
        f"<b>Exploitant(s) :</b> {txt(r.exploitants)}<br>"
        f"<b>Mise en service :</b> {date}<br>"
        f"<b>Antennes :</b> {txt(r.nb_antennes, '0')} ({txt(r.types_antenne)})<br>"
        f"<b>Systèmes :</b> {txt(r.systemes)}"
    )

# On crée la bulle de chaque support
df['popup'] = [popup(r) for r in df.itertuples()]



# 7. LA CARTE

# On crée une carte, centrée sur la France
Carte = folium.Map(
    location=[47, 1],        # Position de départ de la carte
    zoom_start=6,            # Zoom au départ
    tiles=None,              # Pas de fond par défaut
    min_zoom=4,              # Zoom minimum
)

# On ajoute les fonds de carte sans avoir besoin de compte ni de clé

# 1. Esri : les rues fond par défaut
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/'
          'World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles © Esri', name='Esri rues', no_wrap=True,
).add_to(Carte)

# 2. Esri : la vue satellite
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/'
          'World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles © Esri', name='Esri satellite', no_wrap=True, show=False,
).add_to(Carte)

# 3. IGN : la carte officielle française
folium.TileLayer(
    tiles='https://data.geopf.fr/wmts?REQUEST=GetTile&SERVICE=WMTS&VERSION=1.0.0'
          '&STYLE=normal&TILEMATRIXSET=PM&FORMAT=image/png'
          '&LAYER=GEOGRAPHICALGRIDSYSTEMS.PLANIGNV2'
          '&TILEMATRIX={z}&TILEROW={y}&TILECOL={x}',
    attr='© IGN', name='IGN Plan', no_wrap=True, show=False,
).add_to(Carte)

# 4. OpenTopoMap : le relief
folium.TileLayer(
    tiles='https://tile.opentopomap.org/{z}/{x}/{y}.png',
    attr='© OpenTopoMap, © OpenStreetMap', name='OpenTopoMap',
    no_wrap=True, show=False,
).add_to(Carte)

# On affiche combien de supports sont sur la carte
print(f"{len(df)} supports affichés")

if len(df) == 0:

    # Aucun support trouvé
    print("Aucun support : vérifie le département (ex. 76, 27, 2A).")
else:

    # Peu de supports : ronds cliquables regroupés en paquets
    groupe = MarkerCluster(name='Supports').add_to(Carte)
    for r in df.itertuples():
        folium.CircleMarker(
            location=[r.lat, r.lon],
            radius=6,
            color='#c0392b', fill=True, fill_color='#e74c3c', fill_opacity=0.8,
            popup=folium.Popup(r.popup, max_width=350),
        ).add_to(groupe)

# On ajoute le menu pour changer de fond de carte
folium.LayerControl().add_to(Carte)

# On sauvegarde la carte dans un fichier HTML
Carte.save('carte_code-f.html')

# On ouvre la carte dans une page web
webbrowser.open('file://' + os.path.abspath('carte_code-f.html'))

99073 supports affichés


True